# FICHAMENTO CIENTÍFICO — Run-time Prevention of Thermal Throttling on the Edge using Reinforcement-Learning Based Predictive Thermal Aware Power and Performance Management

> **Veredito de Relevância:** **SIM, o artigo é altamente útil para o projeto de AOC.** O documento trata diretamente de *thermal throttling*, DVFS (Dynamic Voltage and Frequency Scaling), gerenciamento de potência (TDP/Power Limit), uso de IPC como métrica de estado da CPU e validação estatística de erro (MAE, desvio padrão amostral) — todos pilares centrais do nosso escopo de Benchmarking, CPU, Energia e Estatística Experimental. Servirá como referência de peso para a Fundamentação Teórica (throttling térmico) e para justificar metodologicamente a coleta contínua de telemetria (paralelo direto ao nosso uso do HWiNFO64).

---

## 1. IDENTIFICAÇÃO BIBLIOGRÁFICA REGULAR

- **Referência Textual Padrão SBC (para `\begin{thebibliography}` do `main.tex`):**

```
NISHA, P.; VINAY, R.; LAAD, K.; SASMAL, P.; HARAKI, T.; JUYAL, C.;
ELBAKRI, M. A. G.; ACHARYYA, A. Run-time Prevention of Thermal
Throttling on the Edge using Reinforcement-Learning Based Predictive
Thermal Aware Power and Performance Management. In: 2024 22nd IEEE
Interregional NEWCAS Conference (NEWCAS). [S.l.]: IEEE, 2024.
p. 273–277. DOI: 10.1109/NEWCAS58973.2024.10666109.
```

- **Código BibTeX Completo (para `sbc-template.bib`):**

```bibtex
@InProceedings{nisha2024thermal,
  author    = {Parveen Nisha and Ratnala Vinay and Kartik Laad and Pradip Sasmal
               and Toshihisa Haraki and Chirag Juyal and Mohamed Amir Gabir Elbakri
               and Amit Acharyya},
  title     = {Run-time Prevention of Thermal Throttling on the Edge using
               Reinforcement-Learning Based Predictive Thermal Aware Power and
               Performance Management},
  booktitle = {2024 22nd {IEEE} Interregional {NEWCAS} Conference ({NEWCAS})},
  publisher = {IEEE},
  year      = {2024},
  pages     = {273--277},
  doi       = {10.1109/NEWCAS58973.2024.10666109}
}
```

> **Nota de verificação de unicidade:** A chave `nisha2024thermal` foi conferida contra o `sbc-template.bib` atual do projeto (que contém apenas `knuth:84`, `boulic:91`, `smith:99`) — não há conflito de chaves.

---

## 2. METADADOS E OBJETIVOS DO DOCUMENTO

- **Grau/Tipo:** Artigo de Conferência (*Conference Paper*) — IEEE NEWCAS 2024 (22nd IEEE Interregional NEWCAS Conference).
- **Instituição/Editora:** IEEE (Institute of Electrical and Electronics Engineers). Autoria vinculada ao IIT Hyderabad, IIT Jodhpur (India) e Suzuki Motor Corporation (Japão).
- **Palavras-Chave Originais (Index Terms):** DVFS (Dynamic voltage and frequency scaling), IPC (Instruction per cycle), PPTM (Power, performance, and thermal management), OS (Operating System), Q-learning, Regression model, Thermal throttling.
- **Resumo do Escopo Geral:** O artigo propõe um algoritmo baseado em *Reinforcement Learning* (Q-learning) combinado com um modelo de regressão linear preditivo de temperatura, para evitar proativamente eventos de *thermal throttling* em sistemas embarcados de borda (placa Nvidia Jetson Nano, processador ARM A57). A proposta é comparada com os governadores nativos do sistema operacional (Performance, Ondemand, Powersave), demonstrando economia média de energia de 11,19% e eliminação total dos eventos de throttling, sem degradação de desempenho, ao custo de um erro médio absoluto de predição de temperatura de apenas 0,28 °C.

---

## 3. FICHAMENTO ESPECÍFICO E DETALHADO (CITAÇÕES DIRETAS E INDIRETAS)

### 3.1 Conceito/Teoria: Thermal Throttling (Estrangulamento Térmico)

- **Citação Direta (Ipsis Litteris):** "Thermal throttling is an event that gets triggered when board temperatures are high which in turn halts all the running applications until the board temperature reaches a safe limit." (Página 1).

- **Paráfrase (Citação Indireta Acadêmica):** O estrangulamento térmico (*thermal throttling*) é definido como o mecanismo de proteção ativado quando a temperatura do processador atinge um limiar crítico, fazendo com que o sistema reduza compulsoriamente a frequência de operação — e, em casos extremos, interrompa temporariamente os processos em execução — até que a temperatura retorne a níveis seguros (NISHA et al., 2024). Esse comportamento reativo, embora proteja o hardware, impõe uma penalidade de desempenho que pode superar 500% em relação à execução na frequência máxima sustentada, conforme demonstrado experimentalmente pelos autores (Página 1).

- **Onde Encaixar no Artigo LaTeX:** Fundamentação Teórica (subseção sobre Termodinâmica e Arquitetura) e Introdução, como justificativa motivacional do problema investigado.

- **Mapeamento de Colunas e Arquivos de Teste:** Esta citação sustenta diretamente o uso das colunas booleanas de throttling presentes nos 80 arquivos `maq*_rodada_*.CSV`: `Estrangulamento térmico do núcleo (avg) (Yes/No)`, `Core 0/1/2/3 Estrangulamento térmico (Yes/No)`, `Regulagem térmica do pacote / anel (Yes/No)` e `IA: Atenuação turbo (MCT) (Yes/No)`. Sempre que essas colunas registrarem "Yes" em qualquer rodada de qualquer máquina, deve-se cruzar o evento com a coluna `Relógios efetivos núcleo (avg) (MHz)` na mesma linha temporal, para evidenciar a queda de clock correspondente, e com o score da rodada equivalente em `scores_maq*.txt` (colunas `Single_Core`/`Multi_Core`), validando empiricamente a relação causal throttling → degradação de score.

---

### 3.2 Conceito/Teoria: Gestão Térmica Reativa vs. Proativa (DVFS)

- **Citação Direta (Ipsis Litteris):** "These management methods try to reduce the temperature once it reaches to system's temperature threshold by shutting down the core or reducing frequency reactively inevitably impacting the system's performance as it fails to prevent thermal throttling from triggering." (Página 1).

- **Paráfrase (Citação Indireta Acadêmica):** Os mecanismos tradicionais de gerenciamento térmico operam de forma reativa: a redução de frequência (DVFS) ou o desligamento de núcleos só é acionada depois que a temperatura já ultrapassou o limiar crítico, o que torna o sistema incapaz de evitar o próprio evento de throttling, apenas mitigando seus efeitos posteriormente (NISHA et al., 2024, p. 1). Em contraposição, a proposta do artigo introduz um componente preditivo que antecipa a temperatura futura, permitindo decisões de escalonamento de frequência antes que o limiar seja atingido.

- **Onde Encaixar no Artigo LaTeX:** Fundamentação Teórica — relacionando com o conceito de DVFS como técnica clássica de gerenciamento de energia em arquiteturas modernas, e na discussão de Resultados ao explicar por que algumas máquinas do nosso experimento podem apresentar quedas abruptas de clock sem aviso prévio no log.

- **Mapeamento de Colunas e Arquivos de Teste:** Sustenta o cruzamento entre `Relógios núcleo (avg) (MHz)` (clock nominal solicitado) e `Relógios efetivos núcleo (avg) (MHz)` (clock real entregue) ao longo do tempo dentro de cada `maq*_rodada_*.CSV`. A diferença abrupta entre essas duas colunas em uma mesma amostra temporal é o indicador estatístico direto da atuação reativa do DVFS, especialmente quando acompanhada por `CPU Inteira (°C)` próxima do limite térmico do fabricante.

---

### 3.3 Conceito/Teoria: Instruction Per Cycle (IPC) como Métrica de Estado

- **Citação Direta (Ipsis Litteris):** "IPC is the ratio of the retired instructions to the total cycles ticked by the CPU." (Página 2).

- **Paráfrase (Citação Indireta Acadêmica):** O IPC (Instruções por Ciclo) é definido pelos autores como a razão entre o número de instruções efetivamente concluídas (retiradas do pipeline) e o número total de ciclos de clock consumidos pela CPU naquele intervalo (NISHA et al., 2024, p. 2). Essa métrica é amplamente reconhecida na literatura clássica de arquitetura de computadores como um dos componentes fundamentais da Lei de Desempenho, complementando a frequência de clock na determinação do tempo de execução efetivo de um programa (HENNESSY; PATTERSON, 2017).

- **Onde Encaixar no Artigo LaTeX:** Fundamentação Teórica — seção de Métricas de Desempenho, articulando com MIPS, FLOPS e Scores Sintéticos do Geekbench 6, e na discussão sobre Eficiência Microarquitetural (IPC Relativo).

- **Mapeamento de Colunas e Arquivos de Teste:** Embora nosso dataset HWiNFO64 não registre IPC diretamente, ele pode ser **aproximado/inferido** combinando `Uso total da CPU (%)`, `Relógios efetivos núcleo (avg) (MHz)` e o score do `scores_maq*.txt` (`Multi_Core`), permitindo discutir o conceito de "IPC Relativo" conforme proposto nas diretrizes do projeto (item 5 do prompt mestre): quanto maior o score obtido para um mesmo clock efetivo médio, maior a eficiência de instruções por ciclo da microarquitetura da máquina em questão.

---

### 3.4 Conceito/Teoria: Modelo de Regressão Linear Múltipla para Predição de Temperatura

- **Citação Direta (Ipsis Litteris):** "A linear regression model has been used to predict the temperature using the previous two temperature values, CPU power, and CPU load." (Página 2).

- **Paráfrase (Citação Indireta Acadêmica):** Os autores empregam um modelo de regressão linear múltipla para estimar a temperatura futura da CPU a partir de quatro variáveis preditoras: as duas últimas leituras de temperatura registradas, a potência instantânea consumida pela CPU e a carga de utilização do processador (NISHA et al., 2024, p. 2). O modelo, treinado em uma faixa de temperatura de 30 °C a 97 °C, alcançou um coeficiente de determinação (R²) de 0,997, validando estatisticamente a robustez da relação entre essas variáveis físicas e o comportamento térmico do processador.

- **Onde Encaixar no Artigo LaTeX:** Fundamentação Teórica e, possivelmente, Metodologia — como inspiração teórica para discutirmos a correlação estatística entre `Potência total da CPU (W)`, `Uso total da CPU (%)` e `CPU Inteira (°C)` em nossos próprios dados, ainda que não estejamos construindo um modelo preditivo, mas sim uma análise descritiva/correlacional retrospectiva.

- **Mapeamento de Colunas e Arquivos de Teste:** Justifica teoricamente a análise de correlação entre as colunas `Potência total da CPU (W)`, `Uso total da CPU (%)` e `CPU Inteira (°C)` / `Núcleo máximo (°C)` nos 80 arquivos CSV. Sugerimos calcular o coeficiente de correlação de Pearson entre essas três séries temporais por máquina, como forma de validar empiricamente (mesmo sem regressão preditiva) a relação carga→potência→temperatura descrita no artigo.

---

### 3.5 Conceito/Teoria: Equação Governante do Modelo de Predição de Temperatura

- **Citação Direta (Ipsis Litteris):** "The corresponding governing equation of the multiple linear regression model is as follows: $T_{pred}(t) = constant + A_0T(t-1) + A_1T(t-2) + A_2CPU_{power} + A_3CPU_{load}$" (Página 2, Equação 1).

- **Paráfrase (Citação Indireta Acadêmica):** A temperatura predita no instante *t* é modelada como uma combinação linear de uma constante, das duas temperaturas anteriores ponderadas por coeficientes de regressão ($A_0$, $A_1$), e das variáveis instantâneas de potência e carga da CPU, ponderadas pelos coeficientes $A_2$ e $A_3$, respectivamente (NISHA et al., 2024, p. 2, Eq. 1). Tais coeficientes são obtidos por treinamento supervisionado sobre dados experimentais reais coletados diretamente dos sensores do hardware.

- **Onde Encaixar no Artigo LaTeX:** Fundamentação Teórica — pode ser citada como exemplo de modelagem preditiva de temperatura na literatura, mesmo que nosso trabalho não implemente regressão, servindo de contraponto teórico ao caráter puramente descritivo/estatístico (médias e desvios padrão) da nossa metodologia.

- **Mapeamento de Colunas e Arquivos de Teste:** As variáveis da equação mapeiam diretamente para `CPU Inteira (°C)` ou `Núcleo máximo (°C)` (termo $T(t-1)$, $T(t-2)$), `Potência total da CPU (W)` ($CPU_{power}$) e `Uso total da CPU (%)` ($CPU_{load}$) em nossos arquivos CSV.

---

### 3.6 Conceito/Teoria: Dynamic Voltage and Frequency Scaling (DVFS) como Espaço de Ação

- **Citação Direta (Ipsis Litteris):** "DVFS in run time can scale the voltage and frequency levels thus creating different power consumption and performance scenarios on the CPU which help us train the RL-based run time manager." (Página 2).

- **Paráfrase (Citação Indireta Acadêmica):** O DVFS é caracterizado como a técnica de escalonamento dinâmico de tensão e frequência que, ao ser ajustada em tempo de execução, gera diferentes cenários de consumo de potência e desempenho, sendo aproveitada como o espaço de ações discretas do algoritmo de aprendizado por reforço proposto (NISHA et al., 2024, p. 2). Essa técnica é historicamente fundamentada na literatura clássica de gerenciamento de energia em sistemas portáteis e embarcados (SIMUNIC et al., 2001, *apud* NISHA et al., 2024).

- **Onde Encaixar no Artigo LaTeX:** Fundamentação Teórica — Hierarquia de Memória/Gargalos de Arquitetura, articulando com a explicação do porquê processadores móveis/ultrabook (como o i5-8265U da Máquina D) variam o clock dinamicamente entre 1,60 GHz base e 3,90 GHz boost.

- **Mapeamento de Colunas e Arquivos de Teste:** Sustenta a análise da variação temporal de `Core 0/1/2/3 Relógio (MHz)` e `Relógios núcleo (avg) (MHz)` ao longo de cada rodada nos arquivos `maq*_rodada_*.CSV`, demonstrando empiricamente o comportamento de DVFS em ação nas máquinas reais do grupo (ex.: subida de clock de 1,60 GHz para próximo de 3,90 GHz no i5-8265U da Máquina D durante o benchmark).

---

### 3.7 Conceito/Teoria: Função de Recompensa Multiobjetivo (Potência, Desempenho e Temperatura)

- **Citação Direta (Ipsis Litteris):** "$Reward = \begin{cases} (P_{max}-P_{avg}); & \text{if } T_{pr}\leq T_{th} \text{ and } L_i\geq L_c \\ (L_i-L_c); & \text{if } T_{pr}\leq T_{th} \text{ and } L_i<L_c \\ (T_{th}-T_{pr}); & \text{if } T_{pr}>T_{th} \end{cases}$" (Página 3, Equação 2).

- **Paráfrase (Citação Indireta Acadêmica):** A função de recompensa do algoritmo de Q-learning é definida por casos: quando a temperatura predita está dentro do limite seguro e o desempenho atende ao critério mínimo exigido, a recompensa é proporcional à economia de potência obtida em relação ao consumo máximo; quando o desempenho fica abaixo do crítico (mas a temperatura ainda é segura), a recompensa passa a priorizar a melhoria de desempenho; e quando a temperatura predita excede o limiar, a recompensa torna-se negativa, penalizando o sistema proporcionalmente ao excesso térmico previsto (NISHA et al., 2024, p. 3, Eq. 2). Esse desenho multiobjetivo equilibra três grandezas físicas concorrentes — energia, desempenho e temperatura — centrais à área de Arquitetura de Computadores.

- **Onde Encaixar no Artigo LaTeX:** Fundamentação Teórica — discussão sobre o trade-off clássico entre Desempenho, Potência e Temperatura (relação direta com a proposta de "Eficiência Microarquitetural: Desempenho por Watt" do nosso prompt mestre, item 5).

- **Mapeamento de Colunas e Arquivos de Teste:** Embasa teoricamente o cálculo da métrica "Desempenho por Watt" do nosso projeto: `score do Geekbench 6 (Single_Core ou Multi_Core)` dividido pela média de `Potência total da CPU (W)` na respectiva rodada — um proxy estático e retrospectivo do mesmo trade-off que a função de recompensa do artigo otimiza dinamicamente.

---

### 3.8 Conceito/Teoria: Validação por Comparação com Governadores Nativos do SO (Performance, Ondemand, Powersave)

- **Citação Direta (Ipsis Litteris):** "On the contrary, in Powersave mode the performance is degraded by 26% and 36% compared to the proposed method as mentioned in Table II." (Página 4).

- **Paráfrase (Citação Indireta Acadêmica):** Os autores demonstram que o modo *Powersave*, embora evite o estrangulamento térmico por operar permanentemente em frequência reduzida, sacrifica significativamente o desempenho (degradação de 26% e 36% nos benchmarks Parsec e Network, respectivamente) em comparação ao método proposto, que mantém desempenho equivalente ao modo *Ondemand* sem incorrer em throttling (NISHA et al., 2024, p. 4). Esse resultado evidencia que a simples redução estática de frequência não constitui uma solução eficiente, reforçando a necessidade de abordagens preditivas dinâmicas.

- **Onde Encaixar no Artigo LaTeX:** Resultados e Discussão — pode fundamentar nossa interpretação caso observemos, em alguma das quatro máquinas, comportamento de clock consistentemente baixo e estável (possível modo de economia de energia ativo no Windows 11), versus máquinas com variação ampla de clock (modo de alto desempenho).

- **Mapeamento de Colunas e Arquivos de Teste:** Sustenta a comparação cruzada entre as quatro máquinas a partir de `Relógios efetivos núcleo (avg) (MHz)` médio e desvio padrão por rodada, relacionando-os ao plano de energia do Windows configurado em cada máquina (informação que deve ser documentada na Metodologia, caso ainda não tenha sido).

---

### 3.9 Conceito/Teoria: Validação Estatística do Modelo Preditivo (MAE, Erro Máximo, Desvio Padrão)

- **Citação Direta (Ipsis Litteris):** "Mean absolute error (MAE), maximum absolute error (AEmax), and maximum standard deviation (AEstd) are compared with state of the art." (Página 3).

- **Paráfrase (Citação Indireta Acadêmica):** A validação da acurácia do modelo preditivo é conduzida por meio de três métricas estatísticas complementares: o erro médio absoluto (MAE), que sintetiza a magnitude média dos desvios entre valor predito e real; o erro máximo absoluto, que identifica o pior caso pontual observado; e o desvio padrão máximo do erro, que quantifica a dispersão/estabilidade da predição ao longo do experimento (NISHA et al., 2024, p. 3). Essa tríade metodológica de validação estatística é diretamente análoga à análise de Média e Desvio Padrão Amostral que fundamenta a comparação de desempenho entre as 20 rodadas de cada máquina no presente trabalho.

- **Onde Encaixar no Artigo LaTeX:** Metodologia — seção de Análise Estatística, justificando teoricamente por que reportamos não apenas a média, mas também o desvio padrão amostral dos scores e das variáveis de telemetria.

- **Mapeamento de Colunas e Arquivos de Teste:** Aplica-se diretamente ao cálculo estatístico das 20 rodadas de cada arquivo `scores_maq*.txt` (colunas `Single_Core` e `Multi_Core`): calcular média e desvio padrão amostral por máquina, e relacionar um desvio padrão elevado a possíveis eventos de throttling capturados nas colunas booleanas correspondentes dos arquivos `maq*_rodada_*.CSV`.

---

### 3.10 Conceito/Teoria: Equação de Atualização da Q-Table (Aprendizado por Reforço)

- **Citação Direta (Ipsis Litteris):** "$Q(s,a)_{i-1} = Q(s,a)_{i-1} + \alpha R(t_i)$" (Página 3, Equação 3).

- **Paráfrase (Citação Indireta Acadêmica):** A atualização incremental da tabela de valores Q (Q-table) do algoritmo de aprendizado por reforço é definida como a soma do valor Q anterior com o produto entre a taxa de aprendizado $\alpha$ e a recompensa obtida no instante $t_i$ (NISHA et al., 2024, p. 3, Eq. 3). O valor de $\alpha$ governa a transição entre as fases de exploração (valores próximos de 1, favorecendo ações aleatórias) e exploração-exploração (valores próximos de 0, favorecendo a ação de maior valor Q já aprendido).

- **Onde Encaixar no Artigo LaTeX:** Fundamentação Teórica — pode ser citada apenas como contextualização do mecanismo de aprendizado utilizado pelo artigo de referência, sem necessidade de implementação prática em nosso projeto (que é puramente experimental/estatístico, não preditivo).

- **Mapeamento de Colunas e Arquivos de Teste:** Não há mapeamento direto com nossas colunas, pois nosso projeto não implementa aprendizado por reforço. Esta equação serve apenas como referência teórica complementar sobre técnicas avançadas de gerenciamento térmico dinâmico, contrastando com a abordagem estática (sem controle ativo) das quatro máquinas analisadas.

---

### 3.11 Conceito/Teoria: Espaço de Ação Discreto do DVFS Limitado pelos Níveis de Frequência do Hardware

- **Citação Direta (Ipsis Litteris):** "Discrete action space was taken which is dependent on the number of frequency levels the embedded board can run, DVFS in run time can scale the voltage and frequency levels thus creating different power consumption and performance scenarios on the CPU." (Página 2).

- **Paráfrase (Citação Indireta Acadêmica):** O espaço de ações do algoritmo é definido como discreto justamente porque está limitado pelo número finito de níveis de frequência que o hardware embarcado é capaz de sustentar (NISHA et al., 2024, p. 2). Esse condicionamento do espaço de decisão pelas capacidades físicas do processador evidencia que arquiteturas com maior amplitude entre clock base e clock boost — e, sobretudo, com múltiplos domínios de frequência independentes (como ocorre em microarquiteturas híbridas com núcleos de desempenho e núcleos de eficiência energética) — possuem um espaço de exploração de DVFS substancialmente mais rico do que processadores homogêneos de núcleo único por domínio de clock.

- **Onde Encaixar no Artigo LaTeX:** Fundamentação Teórica — Paralelismo a Nível de Instrução e Thread / Hierarquia de Microarquitetura, ao introduzir o conceito de arquiteturas híbridas (*big.LITTLE*-like) presentes em parte do nosso hardware real.

- **Mapeamento de Colunas e Arquivos de Teste:** Sustenta a observação diferenciada de `Core 0/1/2/3 Relógio (MHz)` e `Core 0/1/2/3 T0/T1 Relógio efetivo (MHz)` nos arquivos `maq*_rodada_*.CSV`. Em processadores híbridos (ver Máquinas A, B e F na Seção 6 abaixo), espera-se observar amplitudes de variação de clock distintas entre núcleos de desempenho (P-cores) e núcleos de eficiência (E-cores) dentro da mesma rodada — um espaço de ação de DVFS efetivamente mais amplo do que o de processadores com núcleos homogêneos (Máquinas C, D e E).

---

> ### ⚠️ NOTA DE FICHAMENTO PREDITIVO (Hardware das Máquinas A, B e C — Versão Atualizada com Dados Completos das 6 Máquinas)
>
> Os itens 3.6, 3.7, 3.8 e 3.11 acima discutem conceitos de DVFS, trade-off potência/desempenho/temperatura, diferentes perfis de governança de energia e amplitude do espaço de ações de escalonamento de frequência, que se manifestam de forma distinta conforme as características de TDP, número de núcleos, cache L3 e litografia de cada processador.
>
> Com o preenchimento completo da tabela de hardware do grupo (Seção 7 do prompt mestre), esses conceitos teóricos passam a ter correspondência empírica direta com processadores reais de TDP entre 15 W (Máquinas B e C, ultrafinos) e 125 W (Máquina F, desktop), cache L3 entre 4 MB (Máquina C) e 24 MB (Máquina F), e arquiteturas que vão de núcleos homogêneos (Zen+, Zen 3, Whiskey Lake-U) a núcleos híbridos P-core/E-core (Raptor Lake-H, Raptor Lake-P, Raptor Lake). A Seção 6 a seguir detalha, componente a componente, como cada citação acima se aplica concretamente a esse hardware.
>
> **Este trecho teórico e seu respectivo mapeamento de colunas permanecem classificados como fichamento preditivo sempre que a discussão extrapolar os dados efetivamente coletados pelo grupo (ex.: comparações entre máquinas cuja telemetria ainda não tenha sido processada), e só serão utilizados na redação final conforme os dados reais de cada máquina forem efetivamente analisados pelo grupo nas próximas interações.**

---

## 4. ELEMENTOS VISUAIS, FÓRMULAS E EQUAÇÕES

### 4.1 Fórmulas Matemáticas/Físicas em LaTeX Puro

**Equação 1 — Modelo de Regressão Linear para Predição de Temperatura (Página 2):**
```latex
T_{pred}(t) = constant + A_0T(t-1) + A_1T(t-2) + A_2CPU_{power} + A_3CPU_{load}
```

**Equação 2 — Função de Recompensa do Algoritmo Q-learning (Página 3):**
```latex
Reward =
\begin{cases}
(P_{max} - P_{avg}); & \text{se } T_{pr} \leq T_{th} \text{ e } L_i \geq L_c \\
(L_i - L_c); & \text{se } T_{pr} \leq T_{th} \text{ e } L_i < L_c \\
(T_{th} - T_{pr}); & \text{se } T_{pr} > T_{th}
\end{cases}
```

**Equação 3 — Atualização da Q-table (Página 3):**
```latex
Q(s,a)_{i-1} = Q(s,a)_{i-1} + \alpha R(t_i)
```

### 4.2 Sugestão de Gráficos/Tabelas Correspondentes para o Grupo

1. **Gráfico de série temporal (Temperatura vs. Tempo) — análogo à Figura 4 do artigo (Página 4):** Plotar, com Matplotlib, a coluna `CPU Inteira (°C)` ao longo do tempo (`Time`) para cada uma das 20 rodadas de cada máquina, sobrepondo uma linha horizontal de referência no limite térmico TjMax do respectivo processador (informação a ser confirmada para Máquinas A, B e C; já documentado para a Máquina D via `Distância do núcleo para TjMAX (avg) (°C)`). Isso recria visualmente o conceito de "limiar térmico" discutido no artigo.

2. **Gráfico de barras com hastes de erro (Score vs. Máquina) — inspirado na Tabela I do artigo (Página 3):** Barplot com a média de `Multi_Core` (e `Single_Core`) de `scores_maq*.txt` por máquina, com hastes de erro representando o desvio padrão amostral das 20 rodadas — exatamente no padrão sóbrio em tons de cinza exigido pela SBC.

3. **Tabela comparativa de desempenho relativo entre máquinas — inspirada nas Tabelas II e III do artigo (Página 4):** Montar, no `main.tex`, uma tabela de percentual de degradação/ganho de desempenho usando uma máquina como referência (ex.: Máquina D), análoga à lógica "Performance mode as reference / Ondemand mode as reference" empregada pelos autores, mas adaptada para comparar nossas quatro máquinas reais entre si.

4. **Gráfico de dispersão (Scatter Plot) Potência vs. Temperatura:** Inspirado na relação implícita na Equação 1 do artigo, plotar `Potência total da CPU (W)` (eixo X) contra `CPU Inteira (°C)` (eixo Y) para cada rodada, permitindo visualizar e discutir qualitativamente a correlação física entre as duas grandezas em nossos dados reais, mesmo sem ajuste de regressão.

---

## 5. SUGESTÕES DE BUSCA BIBLIOGRÁFICA (Google Acadêmico)

Para encontrar artigos complementares de alto impacto que sustentem as discussões acima:

- `thermal throttling CPU performance degradation`
- `estrangulamento térmico desempenho processador`
- `Dynamic Voltage and Frequency Scaling DVFS survey`
- `escalonamento dinâmico de tensão e frequência processadores`
- `reinforcement learning thermal management embedded systems`
- `IPC instructions per cycle CPU performance metric`
- `instruções por ciclo desempenho microarquitetura`
- `CPU power temperature correlation regression model`
- `TDP thermal design power throttling laptop processor`
- `Q-learning DVFS power performance trade-off`

**Acréscimo — buscas para os tópicos de hardware das 6 máquinas (Seção 6):**

- `hybrid CPU architecture P-core E-core performance`
- `arquitetura híbrida processador núcleos desempenho eficiência`
- `dual channel vs single channel memory bandwidth performance`
- `canais de memória RAM desempenho gargalo Von Neumann`
- `cache L3 size impact IPC performance`
- `tamanho cache L3 desempenho microarquitetura`
- `NVMe SSD vs SATA HDD storage hierarchy benchmark`
- `hierarquia de armazenamento SSD NVMe HDD desempenho`
- `GPU SIMD parallelism data-level parallelism benchmark`
- `paralelismo a nível de dados GPU CPU benchmarking`

---

## 6. CRUZAMENTO COMPONENTE A COMPONENTE COM A TABELA DE HARDWARE DAS 6 MÁQUINAS (ACRÉSCIMO)

> Esta seção é um **acréscimo** ao fichamento original, motivado pelo preenchimento completo da tabela de hardware do grupo (6 máquinas: A-Raony, B-Leandro, C-Cinara, D-Roberta, E-Nauan, F-Nicolas). Nenhum item das Seções 1 a 5 foi alterado. Aqui, cada componente de hardware é cruzado com as citações já fichadas (quando aplicável) e justificado o seu uso na redação final.

### 6.1 Microarquitetura Híbrida P-core/E-core (Máquinas A, B e F — Raptor Lake-H, Raptor Lake-P, Raptor Lake)

- **Componentes envolvidos:** Máquina A (i5-13420H, 4P+4E/12T), Máquina B (i5-1334U, 2P+8E/12T), Máquina F (i5-14600KF, 6P+8E/20T).
- **Citação aplicável:** Item 3.11 (Espaço de Ação Discreto do DVFS) e item 3.6 (DVFS como Espaço de Ação).
- **Justificativa de uso:** As três máquinas com microarquitetura *Raptor Lake* (Intel 7) implementam a topologia híbrida Performance-core/Efficiency-core, que constitui exatamente o cenário de "múltiplos domínios de frequência independentes" mencionado na paráfrase do item 3.11. Para essas máquinas, a discussão de Fundamentação Teórica deve explicitar que `Core 0/1 Relógio (MHz)` e `Core 2/3 Relógio (MHz)` nos arquivos `maq*_rodada_*.CSV` correspondentes podem representar domínios de clock distintos (P-cores vs. E-cores), e que a soma de Cores+Threads (ex.: 4P+4E/12T na Máquina A) não decorre de Hyper-Threading uniforme, mas da arquitetura assimétrica: P-cores com 2 threads cada (Hyper-Threading) e E-cores com 1 thread cada — o que deve ser registrado explicitamente na Metodologia para não induzir erro de interpretação ao leitor.
- **Mapeamento de colunas:** `Core 0/1/2/3 Relógio (MHz)`, `Core 0 T0/T1 Relógio efetivo (MHz)` … `Core 3 T0/T1 Relógio efetivo (MHz)`, `Uso do núcleo (avg) (%)` por core/thread.
- **Nota de fichamento preditivo:** Aplica-se a nota da Seção 3 — esta discussão só será incorporada à redação final quando a telemetria real dessas três máquinas for processada e os clocks por core forem efetivamente segmentados em P-core/E-core nos scripts Python.

### 6.2 TDP Base da CPU — Espectro de 15 W a 125 W (Todas as Máquinas)

- **Componentes envolvidos:** Máquinas B e C (15 W), Máquina D (15 W, já fichada), Máquina A (45 W), Máquina E (65 W), Máquina F (125 W).
- **Citação aplicável:** Item 3.7 (Função de Recompensa Multiobjetivo) e item 3.9 (Validação Estatística do Modelo Preditivo).
- **Justificativa de uso:** A função de recompensa do artigo usa $P_{max}$ (potência na frequência máxima) como termo de referência para a recompensa baseada em economia de energia (Eq. 2, item 3.7). Esse $P_{max}$ é, na prática, limitado pelo TDP de projeto do processador — conceito que agora pode ser concretamente ilustrado pelo contraste entre o TDP de 15 W das Máquinas B e C (ultrafinos) e os 125 W da Máquina F (desktop com i5-14600KF). Isso fundamenta teoricamente a métrica de "Desempenho por Watt" (Seção 5 do prompt mestre): espera-se, por definição arquitetural, que processadores de TDP mais alto entreguem `Potência total da CPU (W)` média substancialmente maior nos arquivos CSV, o que deve ser normalizado pelo score do Geekbench 6 para permitir comparação justa de eficiência energética entre as 6 máquinas.
- **Mapeamento de colunas:** `Potência total da CPU (W)`, `Limite de potência PL1 (Static) (W)`, `Limite de potência PL1 (Dynamic) (W)`, `Limite de potência PL2 (Static) (W)`, `Limite de potência PL2 (Dynamic) (W)` (estas duas últimas só existem nas plataformas Intel — Máquinas A, B, D e F; a Máquina E, AMD Ryzen, não expõe PL1/PL2 da mesma forma, devendo-se documentar essa limitação de telemetria na Metodologia).
- **Nota de fichamento preditivo:** Não se aplica integralmente — o TDP é um dado de catálogo já confirmado para todas as 6 máquinas; o que permanece preditivo é apenas a correlação estatística com a potência real medida, pendente do processamento dos CSVs.

### 6.3 Cache L3 — Amplitude de 4 MB a 24 MB (Máquinas C e F como Extremos)

- **Componentes envolvidos:** Máquina C (4 MB, AMD Ryzen 5 3500U/Zen+), Máquina F (24 MB, i5-14600KF/Raptor Lake).
- **Citação aplicável:** Item 3.3 (IPC como Métrica de Estado) e o conteúdo teórico geral de Hierarquia de Memória previsto no escopo do projeto (item 5 do prompt mestre).
- **Justificativa de uso:** A literatura clássica de Arquitetura de Computadores associa cache L3 maior a menor taxa de *cache miss* para conjuntos de dados de trabalho (working sets) maiores, reduzindo a frequência de acessos à memória principal e, consequentemente, elevando o IPC efetivo (HENNESSY; PATTERSON, 2017). A diferença de 6x entre o cache L3 da Máquina C (4 MB) e da Máquina F (24 MB) cria o par de extremos ideal para o grupo testar empiricamente essa relação teórica: se a Máquina F demonstrar `Multi_Core` (Geekbench 6) proporcionalmente superior ao seu ganho de clock e núcleos em relação à Máquina C, parte dessa vantagem pode ser atribuída ao cache L3, não apenas ao clock ou ao TDP.
- **Mapeamento de colunas:** Não há coluna direta de "cache hit/miss" na lista de telemetria do HWiNFO64 fornecida; a inferência deve ser feita indiretamente via IPC relativo (`Relógios efetivos núcleo (avg) (MHz)` vs. score), conforme já fichado no item 3.3.
- **Nota de fichamento preditivo:** Mantém-se preditivo — depende do cruzamento real entre os scores e a telemetria de clock das Máquinas C e F.

### 6.4 Memória RAM — DDR5 Dual Channel vs. DDR4 Single Channel (Máquinas A e F vs. C e D)

- **Componentes envolvidos:** Máquina A (8 GB DDR5 5200 MT/s, Dual Channel — único caso DDR5 do grupo), Máquina F (32 GB DDR4 3600 MHz, Dual Channel), Máquinas C e D (Single Channel, DDR4 2400 MHz).
- **Citação aplicável:** Não há citação direta do artigo de Nisha et al. (2024) sobre topologia de memória, pois o artigo foca em CPU/temperatura/potência e não aborda largura de banda de memória. **Esta correlação é uma extensão teórica do grupo, não uma citação do artigo fichado.**
- **Justificativa de uso:** Ainda assim, o item 3.4 (Modelo de Regressão para Temperatura) cita `CPU_{load}` como variável de carga — e a carga de CPU pode ser indiretamente amplificada por gargalos de memória (a CPU permanece ociosa aguardando dados, mas isso não necessariamente aparece como queda de `Uso total da CPU (%)` em benchmarks como o Geekbench 6, que são predominantemente CPU-bound). Recomenda-se que esta discussão de Dual Channel DDR5 (Máquina A) vs. Single Channel DDR4 (Máquinas C e D) seja tratada na Fundamentação Teórica como o **Gargalo de Von Neumann** citado no escopo geral do projeto (item 5 do prompt mestre), e não como extensão da teoria de Nisha et al. (2024).
- **Mapeamento de colunas:** `Relógio da memória (MHz)`, `Relação do relógio da memória (x)`, `Tcas (T)`, `Trcd (T)`, `Trp (T)`, `Tras (T)`, `Trc (T)`, `Trfc (T)`, `Command Rate (T)`.
- **Nota de fichamento preditivo:** Aplica-se a nota da Seção 3 — discussão pendente do processamento real dos CSVs das Máquinas A, C, D e F.

### 6.5 Armazenamento — HDD SATA (Máquina D) vs. SSD NVMe PCIe Gen 4.0 (Máquina A) vs. SSD SATA + HDD (Máquina E)

- **Componentes envolvidos:** Máquina D (HDD WD Blue 1TB, 5400 RPM, SATA III), Máquina A (SSD NVMe PCIe Gen 4.0 x4), Máquina E (SSD SATA 120GB + HDD 1TB).
- **Citação aplicável:** Não há citação direta do artigo de Nisha et al. (2024) sobre subsistema de armazenamento — **extensão teórica do grupo**, não citação do artigo fichado.
- **Justificativa de uso:** O artigo não trata de I/O de disco, mas a métrica de "carregamento dos testes" prevista no escopo do projeto (item 5 do prompt mestre, "Hierarquia de Memória") deve ser tratada à parte, com base em literatura própria de hierarquia de armazenamento (a ser buscada separadamente — ver sugestões de busca abaixo). É relevante notar, porém, que a Máquina E mistura SSD SATA (sistema/aplicações) com HDD secundário, cenário que deve ser identificado explicitamente na Metodologia para que os resultados do Geekbench 6 dessa máquina sejam interpretados em função do disco de instalação do benchmark, não do HDD secundário.
- **Mapeamento de colunas:** `Taxa de leituras (MB/s)`, `Taxa de gravações (MB/s)`, `Atividade de leitura (%)`, `Atividade de gravação (%)`, `Temperatura do disco (°C)`.
- **Nota de fichamento preditivo:** Não se aplica à teoria de Nisha et al. (2024), por estar fora do escopo do artigo; trata-se de lacuna bibliográfica a ser preenchida com fontes adicionais.

### 6.6 GPU Dedicada — Da Ausência (Máquinas B e C) ao RTX 4050 Laptop (Máquina A) e RX 7600 (Máquina E)

- **Componentes envolvidos:** Máquinas B e C (sem GPU dedicada), Máquina A (RTX 4050 Laptop, PCIe 4.0 x8), Máquina E (RX 7600, PCIe 4.0 x8), Máquina F (RTX 3050 8GB).
- **Citação aplicável:** O artigo de Nisha et al. (2024) não aborda GPU — menciona apenas CPU A57 (ARM). **Extensão teórica do grupo.**
- **Justificativa de uso:** Como o Geekbench 6 também produz scores de GPU (Compute), recomenda-se — se o grupo optar por incluir essa métrica — tratar a comparação entre Máquinas A, E e F (com GPU dedicada) e B, C e D (sem GPU dedicada ou apenas com MX130 de baixa performance, no caso da Máquina D) como uma discussão de Paralelismo a Nível de Dados (SIMD em massa via GPU), distinta da discussão de Paralelismo a Nível de Instrução/Thread (multicore CPU) já fichada no item 3.3.
- **Mapeamento de colunas:** `GPU Clock (MHz)`, `Carga do núcleo da GPU (%)`, `GPU Temperatura (°C)`, `Potência das linhas GPU (avg) (W)`, `Limite térmico da GPU (°C)`.
- **Nota de fichamento preditivo:** Aplica-se a nota da Seção 3, condicionada à decisão do grupo de incluir ou não benchmark de GPU no escopo final do artigo.

### 6.7 Plano de Energia do Windows 11 — Home vs. Pro (Máquina F)

- **Componentes envolvidos:** Máquina F é a única do grupo com Windows 11 **Pro** (25H2); as demais cinco rodam Windows 11 **Home**.
- **Citação aplicável:** Item 3.8 (Validação por Comparação com Governadores Nativos do SO).
- **Justificativa de uso:** O artigo de Nisha et al. (2024) compara governadores de energia (Performance, Ondemand, Powersave) do sistema operacional embarcado (item 3.8). Embora o Windows não exponha governadores idênticos aos do Linux, a edição **Pro** do Windows disponibiliz a opções de gerenciamento de energia e configurações de grupo de política adicionais não presentes na edição Home, o que deve ser documentado como variável de controle na Metodologia: a Máquina F, além de possuir o maior TDP (125 W) do grupo, também é a única com acesso a controles de energia potencialmente mais granulares do SO, fator que deve ser mencionado como limitação/variável a controlar ao comparar diretamente seus resultados de desvio padrão com os das demais máquinas.
- **Mapeamento de colunas:** Não há coluna telemétrica direta para "edição do SO"; esta é uma variável de documentação metodológica, não de dataset.
- **Nota de fichamento preditivo:** Aplica-se a nota da Seção 3 — a influência real (se houver) só poderá ser confirmada após a análise estatística comparativa entre a Máquina F e as demais.

---

*Fichamento elaborado em conformidade com as Diretrizes de Fichamento de AOC do projeto. Nenhum dado quantitativo foi inventado; todas as citações, números e equações refletem estritamente o conteúdo original do artigo de Nisha et al. (2024). A Seção 6 foi acrescentada nesta atualização para cruzar as citações já fichadas (Seções 1 a 5, inalteradas) com a tabela completa de hardware das 6 máquinas do grupo (A-Raony, B-Leandro, C-Cinara, D-Roberta, E-Nauan, F-Nicolas), explicitando onde cada componente sustenta, estende ou extrapola (fichamento preditivo) a teoria do artigo original.*
